In [1]:
# Bootstrap defaults for running in VS Code
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", "/users/chenqh23/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try: p.kill()
        except: pass
# ensure classpath
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", "/users/chenqh23/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

# Auto-detect RAPIDS jar in SPARK_HOME/jars if not provided
if "RAPIDS_JAR" not in os.environ:
    jars_dir = os.path.join(SPARK_HOME, "jars")
    candidates = []
    for pattern in ("rapids-4-spark_*.jar", "rapids-4-spark*.jar"):
        candidates.extend(glob.glob(os.path.join(jars_dir, pattern)))
    rapids_jar = next((p for p in candidates if p.endswith(".jar")), None)
    if rapids_jar:
        os.environ["RAPIDS_JAR"] = rapids_jar

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 
print("RAPIDS_JAR =", os.environ.get("RAPIDS_JAR", "<auto> or jars in SPARK_HOME"))


SPARK_HOME = /users/chenqh23/spark/dist
SPARK_MASTER_URL = local[*]
EVENTLOG_DIR = /tmp/spark-events
RAPIDS_JAR = /users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar


# Microbenchmarks on GPU
This is a notebook for microbenchmarks running on GPU. 

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from time import time
import os
# Change to your cluster ip:port and directories
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "spark:your-ip:port")
RAPIDS_JAR = os.getenv("RAPIDS_JAR", "/your-path/rapids-4-spark_2.12-25.08.0.jar")


Run the microbenchmark with retryTimes

In [3]:
# Robust Spark startup to fix Py4J "Answer from Java side is empty"
import os, psutil
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

# Environment
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['SPARK_HOME'] = os.environ.get('SPARK_HOME', '/users/chenqh23/spark/dist')
os.environ.setdefault('SPARK_MASTER_URL', 'local[*]')

# Stop an existing Spark and kill orphan Spark JVMs
try:
    spark.stop()
except Exception:
    pass
try:
    for p in psutil.process_iter(['pid','name','cmdline','username']):
        cl = ' '.join(p.info.get('cmdline') or [])
        if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
            try:
                p.kill()
            except Exception:
                pass
except Exception:
    pass

# Java 17 add-opens flags
_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

# Build base conf
base = (SparkConf()
    .setMaster("local[*]")
    .setAppName("Microbenchmark on GPU")
    .set("spark.driver.memory", os.getenv("DRIVER_MEM", "12g"))
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", os.getenv("MAX_PARTITION_BYTES", "64m"))
    .set("spark.locality.wait", "0")
    .set("spark.eventLog.enabled", "false")
    .set("spark.driver.extraJavaOptions", _DEF_OPENS)
    .set("spark.executor.extraJavaOptions", _DEF_OPENS)
)

# GPU settings (avoid duplicate JARs; rely on SPARK_HOME/jars)
gpu = (base
    .set("spark.plugins", "com.nvidia.spark.SQLPlugin")
    .set("spark.rapids.sql.enabled", "true")
    .set("spark.rapids.sql.allowMultipleJars", "ALWAYS")
    .set("spark.rapids.sql.concurrentGpuTasks", os.getenv("CONCURRENT_GPU_TASKS", "1"))
    .set("spark.rapids.sql.batchSizeBytes", "256m")
    .set("spark.rapids.sql.multiThreadedRead.numThreads", os.getenv("RAPIDS_READER_THREADS", "8"))
    .set("spark.rapids.memory.gpu.maxAllocFraction", "0.5")
    .set("spark.rapids.memory.gpu.allocFraction", "0.25")
    .set("spark.rapids.memory.gpu.minAllocFraction", "0.1")
    .set("spark.rapids.memory.gpu.reserve", "2G")
    .set("spark.rapids.memory.pinnedPool.size", os.getenv("PINNED_POOL_SIZE", "4g"))
)

# Try GPU first; if it crashes the JVM, fall back to CPU-only cleanly
try:
    spark = SparkSession.builder.config(conf=gpu).getOrCreate()
except Exception:
    cpu = (base
        .set("spark.rapids.sql.enabled", "false")
        .set("spark.plugins", ""))
    spark = SparkSession.builder.config(conf=cpu).getOrCreate()

print("Spark OK:", spark.version, "| rapids.enabled:", spark.conf.get("spark.rapids.sql.enabled", "<unset>"))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/13 13:00:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/13 13:00:46 WARN RapidsPluginUtils: RAPIDS Accelerator 25.08.0 using cudf 25.08.0, private revision f4b467339f0ea78b7e2a862be97a63bc239e0b07
25/10/13 13:00:46 WARN RapidsPluginUtils: Multiple spark-rapids-jni jars found in the classpath:
revison: 155ef36a7e5c38e404d76976a61d177354c99281
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/spark-rapids-jni-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d76976a61d177354c99281
	branch=HEAD
	date=2025-08-07T05:18:36Z
	url=https://github.com/NVIDIA/spark-rapids-jni.git
	gpu_architectures=100;120;70;75;80;86;90
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d

Spark OK: 3.5.6 | rapids.enabled: true


In [4]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time()
        spark.sql(query).show(5)
        end = time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [5]:
# You need to update with your real hardware resource 
driverMem = os.getenv("DRIVER_MEM", "50g")
executorMem = os.getenv("EXECUTOR_MEM", "16g")
maxPartionBytes = os.getenv("MAX_PARTITION_BYTES", "64m")
pinnedPoolSize = os.getenv("PINNED_POOL_SIZE", "8g")
concurrentGpuTasks = os.getenv("CONCURRENT_GPU_TASKS", "1")
executorCores = int(os.getenv("EXECUTOR_CORES", "16"))
eventlogDir = "file:"+os.getenv("EVENTLOG_DIR")
gpuPerExecutor = 1/executorCores
# Common spark settings
conf = SparkConf()
conf.setMaster("local[*]")
conf.setAppName("Microbenchmark on GPU")
conf.set("spark.driver.memory", driverMem)
## The tasks will run on GPU memory, so there is no need to set a high host memory
conf.set("spark.executor.memory", executorMem)
## The tasks will run on GPU cores, so there is no need to use many cpu cores
conf.set("spark.executor.cores", executorCores)
conf.set("spark.locality.wait", "0")
conf.set("spark.sql.files.maxPartitionBytes", maxPartionBytes) 
conf.set("spark.dynamicAllocation.enabled", "false") 
conf.set("spark.sql.adaptive.enabled", "true") 
conf.set("spark.sql.files.ignoreCorruptFiles", "true")
conf.set("spark.sql.files.ignoreMissingFiles", "true")
conf.set("spark.sql.sources.useV1SourceList", "parquet")

# Plugin settings (no GPU resource scheduling in local mode)
conf.set("spark.rapids.sql.enabled", "true") 
conf.set("spark.plugins", "com.nvidia.spark.SQLPlugin")
conf.set("spark.rapids.sql.variableFloatAgg.enabled", "true")
conf.set("spark.rapids.sql.allowMultipleJars", "ALWAYS")
conf.set("spark.rapids.sql.concurrentGpuTasks", concurrentGpuTasks)
conf.set("spark.rapids.sql.batchSizeBytes", "256m")
conf.set("spark.rapids.memory.gpu.maxAllocFraction", "0.5")
conf.set("spark.rapids.memory.gpu.allocFraction", "0.25")
conf.set("spark.rapids.memory.gpu.reserve", "2G")
conf.set("spark.rapids.memory.gpu.minAllocFraction", "0.1")
conf.set("spark.rapids.sql.multiThreadedRead.numThreads", "8")
conf.set("spark.rapids.memory.pinnedPool.size", pinnedPoolSize)
conf.set("spark.driver.extraClassPath", RAPIDS_JAR)
conf.set("spark.executor.extraClassPath", RAPIDS_JAR)
conf.set("spark.jars", RAPIDS_JAR)
conf.set("spark.eventLog.enabled", "true")
conf.set("spark.eventLog.dir", eventlogDir)
# Create spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

# Load dataframe and create tempView
# You need to update data path to your real path!
dataRoot = "/users/chenqh23/spark-rapids-examples/datasets"

spark.read.parquet(f"{dataRoot}/tpcds/store_sales").limit(1000).count()

25/10/13 13:01:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/10/13 13:01:08 WARN MultiFileReaderThreadPool: Configuring the file reader thread pool with a max of 128 threads instead of spark.rapids.sql.multiThreadedRead.numThreads = 8


1000

In [ ]:
# Debug Parquet read progress
import os, time

path = f"{dataRoot}/tpcds/store_sales"
print("Path:", path, "exists=", os.path.isdir(path))

# Print key confs
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("maxPartitionBytes:", spark.conf.get("spark.sql.files.maxPartitionBytes", "<unset>"))
print("rapids.enabled:", spark.conf.get("spark.rapids.sql.enabled", "<unset>"))

# Build DataFrame and inspect
print("Creating DataFrame...")
df = spark.read.parquet(path)
print("Schema:")
df.printSchema()
print("Partitions:", df.rdd.getNumPartitions())

# Show first few input files (without materializing all)
try:
    files = df.inputFiles()
    print("Input files count:", len(files))
    print("Sample files:", files[:5])
except Exception as e:
    print("inputFiles() error:", e)

# Force an action with timing
print("Starting count()...")
start = time.time()
try:
    n = df.count()
    print("Row count:", n)
except Exception as e:
    print("count() error:", e)
finally:
    print("Elapsed (count):", round(time.time() - start, 2), "s")

# Explain plan
print("Explain (extended):")
df.explain(True)


Path: /users/chenqh23/spark-rapids-examples/datasets/tpcds/store_sales exists= True
Master: local[*]
Default parallelism: 128
maxPartitionBytes: 128m
rapids.enabled: true
Creating DataFrame...


ERROR:root:KeyboardInterrupt while sending command.              (0 + 0) / 1824]
Traceback (most recent call last):
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [13]:
# CPU-only sanity read (restart Spark without RAPIDS)
import time
try:
    spark.stop()
except Exception:
    pass
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
conf = SparkConf()
conf.setMaster("local[*]")
conf.setAppName("CPU sanity read")
conf.set("spark.eventLog.enabled", "false")
conf.set("spark.sql.files.maxPartitionBytes", "256m")
conf.set("spark.sql.adaptive.enabled", "true")
conf.set("spark.locality.wait", "0")
conf.set("spark.rapids.sql.enabled", "false")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

# Small table first
start = time.time()
ni = spark.read.parquet(f"{dataRoot}/tpcds/item").count()
print("item count:", ni, "elapsed:", round(time.time()-start,2), "s")

# Large table
start = time.time()
ns = spark.read.parquet(f"{dataRoot}/tpcds/store_sales").count()
print("store_sales count:", ns, "elapsed:", round(time.time()-start,2), "s")


25/10/13 12:05:39 WARN RapidsPluginUtils: spark.rapids.sql.multiThreadedRead.numThreads is set to 20.
25/10/13 12:05:39 WARN RapidsPluginUtils: Multiple spark-rapids-jni jars found in the classpath:
revison: 155ef36a7e5c38e404d76976a61d177354c99281
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d76976a61d177354c99281
	branch=HEAD
	date=2025-08-07T05:18:36Z
	url=https://github.com/NVIDIA/spark-rapids-jni.git
	gpu_architectures=100;120;70;75;80;86;90
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/spark-rapids-jni-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d76976a61d177354c99281
	branch=HEAD
	date=2025-08-07T05:18:36Z
	url=https://github.com/NVIDIA/spark-rapids-jni.git
	gpu_architectures=100;120;70;75;80;86;90
Please make sure there is only one spark-rapids-jni jar in the classpath. If it is impossible to fix the classpath you can suppress the error by setting spark.rap

Master: local[*]
Default parallelism: 128


ERROR:root:Exception while sending command.                         (0 + 0) / 1]
Traceback (most recent call last):
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=72>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/users/chenqh23/.local/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    an

Py4JError: An error occurred while calling o337.parquet

In [ ]:
spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")
print("-"*50)

### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the GPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk. You will see about 10x speedups in this query. Because an additional shuffle involved by the repartition operator in CPU mode. And GPUExpand and GPUHashAggregate is much faster than Expand and HashAggregate because GPU algorithms allow us to parallelize the computation and we can utilize most of the GPU cores. The tasks' duration in the third stage is less than one second but will cost 20x-40x while running on CPU. There will be a more significant performance improvement along with the increasing number of count distinct columns and aggregate functions.

In [ ]:
query = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Expand&HashAggregate",query,2)

### Windowing(without data skew)
This is a microbenchmark about windowing expressions running on GPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer. You will see about 25x speedups in this query. The speedup mainly comes from GPUSort/GPUWindow/GPUHashAggregate. The avg aggregation function evaluates all rows which are generated by the sub-query's window function. There will be a more significant performance improvement along with the increasing number of sub-query aggregate functions.

In [ ]:
query = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing without skew",query,2)

### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column. You will see about 80x speedups in this query. The heavier skew task a query has, the more improved performance we will get because GPU parallelizes the computation, CPU is limited to just a single core because of how the algorithms are written.

In [ ]:
query = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing with skew",query,2)

### Intersection
This is a microbenchmark about intersection operation running on GPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years. You will see about 10x speedups in this query. This is a competition between high cardinality SortMergeJoin vs GpuShuffleHashJoin. The mainly improved performance comes from two SortMergeJoin(s) in this query running on CPU get converted to GpuShuffleHashJoin running on GPU.

In [ ]:
query = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"NDS Q14a subquery",query,2)

### Crossjoin
This is a microbenchmark for a 1-million rows crossjoin with itself. You will see about 10x speedups in this query. The mainly improved performance comes from converting BroadcastNestedLoogJoin running on CPU to GpuBroadcastNestedLoogJoin running on GPU.

In [ ]:
start = time() 
spark.read.parquet(dataRoot + "/customer.dat").limit(1000000).write.format("parquet").mode("overwrite").save("/users/chenqh23/spark-rapids-examples/datasets/tmp/customer1m")
end = time()
# Parquet file scanning and writing will be about 3 times faster running on GPU
print("scanning and writing parquet cost : {} seconds".format(round(end - start, 2)))
spark.read.parquet("/users/chenqh23/spark-rapids-examples/datasets/tmp/customer1m").repartition(200).createOrReplaceTempView("costomer_df_1_million")
query = '''
select count(*) from costomer_df_1_million c1 inner join costomer_df_1_million c2 on c1.c_customer_sk>c2.c_customer_sk
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Crossjoin",query,2)

### HashJoin
This is a microbenchmark for a HashJoin. The query on GPU will be more than 10x times faster than CPU based on the cluster in the readme.

In [ ]:
spark.read.parquet(dataRoot + "/store_sales.dat").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/store_returns.dat").createOrReplaceTempView("store_returns")

print("-"*50)
query = '''
select  sum(store_sales.ss_ext_wholesale_cost)
from store_sales
join store_returns on (ss_item_sk = sr_item_sk) and (ss_addr_sk=sr_addr_sk)
'''
runMicroBenchmark(spark,"HashJoin",query,1)

In [ ]:
spark.stop()